In [ ]:
import os

import pandas as pd
from modules.largus import Largus

In [ ]:
largus = Largus()

In [ ]:
def load_and_concatenate_csvs(folder_path):
    dataframes = []
    
    # Parcourir tous les fichiers dans le dossier et les sous-dossiers
    for root, dirs, files in os.walk(folder_path):
        for filename in files:
            if filename.endswith('.csv'):
                file_path = os.path.join(root, filename)
                if not os.path.exists(file_path):
                    print(f"Found CSV file: {file_path}")
                    
                _df = pd.read_csv(file_path)
                dataframes.append(_df)
            

    # Vérifier si la liste des DataFrames est vide
    if not dataframes:
        raise ValueError("No CSV files found in the specified folder.")

    # Concaténer tous les DataFrames en un seul
    final_df = pd.concat(dataframes, ignore_index=True)
    return final_df

In [ ]:
folder_version_path = "../Data/Versions"
df_raw = load_and_concatenate_csvs(folder_version_path)

In [ ]:
df = df_raw.copy()

In [ ]:
df.head(10)

In [ ]:
df.shape

In [ ]:
df.drop(['Année'], axis=1, inplace=True)

In [ ]:
df.head(10)

In [ ]:
df.tail(10)

In [ ]:
df.isnull().sum()

In [ ]:
def cleaned(dataframe):
    df_cleaned = dataframe.dropna(subset=['Version'])
    return df_cleaned

In [ ]:
import re

In [ ]:
# Fonction pour extraire l'année de l'URL et convertir en int
def process_year(dataframe):
    def extract_year(url):
        match = re.search(r'/(\d{4})/', url)
        if match:
            return int(match.group(1))
        return None

    dataframe['Annee'] = dataframe.apply(
        lambda row: extract_year(row['Url']) if pd.isna(row['Annee']) else row['Annee'],
        axis=1
    )
    dataframe['Annee'] = dataframe['Annee'].astype(int)
    return dataframe

def process_label(dataframe):
    # Fonction pour extraire le modèle de voiture
    def extract_car_label(url):
        match = re.search(r'https://www.largus.fr/(.*)', url)
        if match:
            model_part = match.group(1)
            model = model_part.split('/')[6].split('-')[0]
            model = model.replace('+', ' ')
            return model
        return None

    # Appliquer la fonction si la colonne 'Version' est NaN
    dataframe.loc[dataframe['Version'].isna(), 'Version'] = dataframe.loc[dataframe['Version'].isna(), 'Url'].apply(extract_car_label)

    return dataframe

In [ ]:
df[df['Version'].isna()].tail()

In [ ]:
df.to_csv('../Data/Version_Technical_Details_Raw.csv', index=False)

## Cleaned

In [ ]:
df_processed = (df.pipe(process_year)
                .pipe(process_label)
                )

In [ ]:
df_processed.isna()

In [ ]:
df_processed[df_processed['Version'].isna()]

In [ ]:
df_processed.dropna(subset=['Annee'], inplace=True)

In [ ]:
df_processed = df_processed.sample(frac=1).reset_index(drop=True)

In [ ]:
df_processed

In [ ]:
df_processed.to_csv('../Data/Version_Technical_Details_Raw.csv', index=False)

In [ ]:
df_processed.shape

In [ ]:
path_versions = '../Data/Version_Technical_Details_Raw.csv'
df = pd.read_csv(path_versions)
df.head(10)

In [ ]:
df.shape

In [ ]:
df.to_csv('../Data/Version_Technical_Details.csv', index=False)

In [ ]:
path_old_file = '../Data/Version_Technical_Details.csv'
path_new_file = '../Data/Version_Technical_Details_Raw.csv'

In [ ]:
df_old = pd.read_csv(path_old_file)
df_new = pd.read_csv(path_new_file)

In [ ]:
df_old.shape, df_new.shape

In [ ]:
df_old[df_old['Traiter'] == 1]

In [ ]:
df_new.head(10)

In [ ]:
df_new['Traiter'] = df_new['Url'].map(df_old.set_index('Url')['Traiter']).fillna(0).astype(int)

In [ ]:
df_new.head()

In [ ]:
df_new.shape

In [ ]:
df_new[df_new['Traiter'] == 0]

In [ ]:
df_melange = df_new.sample(frac=1).reset_index(drop=True)

In [ ]:
df_melange


In [ ]:
df_trie = df_melange.sort_values(by='Traiter', ascending=False).reset_index(drop=True)

In [ ]:
df_trie

In [ ]:
df_trie.to_csv('../Data/Version_Technical_Details.csv', index=False)